# Stage 8 - track stitching and merge repair

Stage 8 consumes the latest saved Stage 7 `tracks.csv`. If Stage 7 ran in apply mode, these are the graph-optimized tracks. Stage 8 algorithms are unchanged.

In [ ]:
from src.io import PipelinePaths

SAMPLE_ID = "44b6_0113de3b"
paths = PipelinePaths.discover()
sample_path = paths.sample_zarr(SAMPLE_ID)

In [ ]:
from src.api import run_track_stitching
from src.io import (
    load_processed_dataset_inputs,
    load_stage7_outputs,
    save_stitching_result,
)

inputs = load_processed_dataset_inputs(SAMPLE_ID, paths=paths)
stage7 = load_stage7_outputs(paths=paths)
stage7_graph_metadata = stage7.metadata.get("graph_tracking", {})
print("Stage 7 source directory:", paths.stage7_tracking)
print("Stage 7 graph mode:", stage7_graph_metadata.get("mode", "unknown"))
print("Stage 7 graph algorithm:", stage7_graph_metadata.get("algorithm", "unknown"))
print("Stage 7 tracks loaded:", stage7.tracks["track_id"].nunique())

stitching = run_track_stitching(
    stage7.tracks,
    list(inputs.time_frames),
    inputs.segmentation_files,
    sample_id=SAMPLE_ID,
)
save_stitching_result(stitching, paths.stage8_stitching)
print("Saved Stage 8 output directory:", paths.stage8_stitching)